
# 練習問題: GPU上での台数効果を測定する

## 目標

`#pragma omp target teams distribute parallel for` (Fortran では `!$omp target teams distribute parallel do` ... `!$omp end ...`) を1つ挿入して計算をGPUにオフロードし, チーム数・スレッド数を増やすとどれだけ速くなるか(台数効果)を測定する.

## 課題

`gpu_speedup.cpp` (または `gpu_speedup.f90`) は, `lin_rec` (`x = a*x + b` を `n` 回繰り返す関数) を `m` 個の独立な要素について計算するプログラムである.
現状ではオフロードの指示行が外してあり, 計算本体がホスト(CPU)上で逐次に実行されてしまう.

コメント `TODO` の指示に従って **オフロードの指示行を1つ追加** し, ループをGPU上で並列実行させよ.

- C++: `for` 文の直前に
  `#pragma omp target teams distribute parallel for num_teams(nteams) num_threads(nthreads) map(tofrom: x[0:m])`
  を1行加える.
- Fortran: `do` ループを
  `!$omp target teams distribute parallel do num_teams(nteams) num_threads(nthreads) map(tofrom: x)`
  と `!$omp end target teams distribute parallel do` で囲む.

結果を配列 `x` でCPUに戻して検算するので, `map(tofrom: x...)` が必要であることに注意.
それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=gpu gpu_speedup.cpp -o gpu_speedup.exe

# Fortran
nvfortran -fast -mp=gpu gpu_speedup.f90 -o gpu_speedup.exe
```

実行は

```
OMP_NUM_TEAMS=nteams OMP_NUM_THREADS=nthreads ./gpu_speedup.exe m n
```

とすると, チーム数=`nteams`, スレッド数=`nthreads` で実行する.
`m`, `n` を省略すると `m` = `nteams` × `nthreads`, `n` = 100×1000×1000 となる.

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入する.
`OMP_NUM_THREADS` は 1 または 32 の倍数でなければならないことに注意 (GPUのスレッドは32本単位(ワープ)で動くため).

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=1 OMP_NUM_THREADS=1 ./gpu_speedup.exe
```

スレッド数を変えて一気に測るには:

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

for th in 1 32 64 128 256 512 1024 ; do
    echo -n "$th "
    OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${th} ./gpu_speedup.exe | grep GFLOPS
done
```

チーム数を変えて測るには (スレッド数は上で良かった値に固定):

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

th=最適なスレッド数
for tm in 1 2 4 8 16 32 64 108 ; do
    echo -n "$((tm * th)) "
    OMP_NUM_TEAMS=${tm} OMP_NUM_THREADS=${th} ./gpu_speedup.exe | grep GFLOPS
done
```

## 期待される結果

正しくオフロードできていれば検算で `OK` が表示され, `GFLOPS` の値が出力される.

- `OMP_NUM_TEAMS=1` に固定し `OMP_NUM_THREADS` を 1, 32, 64, ... と増やすと, ある程度まではGFLOPSが向上する.
- 次にスレッド数を固定して `OMP_NUM_TEAMS` を増やすと, このGPU (NVIDIA A100, 108個のStreaming Multiprocessor) ではおおむね108程度までGFLOPSが向上し, それ以降は頭打ちになる.

チーム数・スレッド数が少なすぎるとGPUの演算器を使い切れず性能が出ないこと, 両方を十分大きくして初めてGPUの性能を引き出せることを確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ gpu_speedup.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* x = ax + b をひたすら n 回繰り返す.
   (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
   n 回 mul + add を行う (-> 2 n flops) */
double lin_rec(double a, double b, double c, long n) {
  double x = c;
  for (long j = 0; j < n; j++) {
    x = a * x + b;
  }
  return x;
}

int main(int argc, char ** argv) {
  /* チーム数・スレッド数を環境変数から取得 */
  char * nteams_   = getenv("OMP_NUM_TEAMS");
  int    nteams    = (nteams_   ? atoi(nteams_)   : 1);
  char * nthreads_ = getenv("OMP_NUM_THREADS");
  int    nthreads  = (nthreads_ ? atoi(nthreads_) : 1);
  long m = (1 < argc ? atol(argv[1]) : (long)nteams * nthreads);
  long n = (2 < argc ? atol(argv[2]) : 100 * 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);
  printf("num_teams = %d, num_threads = %d\n", nteams, nthreads);
  printf("m = %ld, n = %ld\n", m, n);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体. 現状では指示行が無いのでCPU上で逐次に実行される. */
  // TODO: 下の for 文の直前に #pragma omp target teams distribute parallel for num_teams(nteams) num_threads(nthreads) map(tofrom: x[0:m]) を1行追加し, ループをGPU上で並列実行させよ. (結果 x をCPUに戻して検算するので map(tofrom: x[0:m]) が必要)
  for (long i = 0; i < m; i++) {
    x[i] = lin_rec(0.99, i + 1, 1.0, n);
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */
  /* 答え確認 (x[i] = 100 * (i + 1) くらいのはず) */
  long err = 0;
  for (long i = 0; i < m; i++) {
    if (fabs(x[i] - 100 * (i + 1)) > 1.0e-3) {
      printf("x[%3ld] = %9.3f\n", i, x[i]);
      err++;
    }
  }
  if (err == 0) printf("OK\n");
  double flops = 2. * (double)m * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("elapsed/m  : %7.3f msec\n", dt / m * 1e3);
  printf("elapsed/n  : %7.3f nsec\n", dt / n * 1e9);
  printf("elapsed/mn : %7.3f nsec\n", dt / (m * n) * 1e9);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu gpu_speedup.cpp -o gpu_speedup_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_speedup_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ gpu_speedup.f90
module lin_rec_mod
contains
  ! x = ax + b をひたすら n 回繰り返す.
  ! (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
  ! n 回 mul + add を行う (-> 2 n flops)
  !$omp declare target
  function lin_rec(a, b, c, n) result(x)
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    real(8) :: x
    integer(8) :: j
    x = c
    do j = 1, n
       x = a * x + b
    end do
  end function lin_rec
end module lin_rec_mod

program gpu_speedup
  use omp_lib
  use lin_rec_mod
  character(len=32) :: arg, env
  integer(8) :: m, n, i
  integer :: nteams, nthreads, st
  integer(8) :: err
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  ! チーム数・スレッド数を環境変数から取得
  nteams = 1
  call get_environment_variable("OMP_NUM_TEAMS", env, status=st)
  if (st == 0) read (env, *) nteams
  nthreads = 1
  call get_environment_variable("OMP_NUM_THREADS", env, status=st)
  if (st == 0) read (env, *) nthreads
  m = int(nteams, 8) * int(nthreads, 8)
  n = 100 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  allocate(x(m))
  print "(a,i0,a,i0)", "num_teams = ", nteams, ", num_threads = ", nthreads
  print "(a,i0,a,i0)", "m = ", m, ", n = ", n
  ! 計測開始
  t0 = omp_get_wtime()
  ! 計算本体. 現状では指示行が無いのでCPU上で逐次に実行される.
  ! TODO: 下の do ループを !$omp target teams distribute parallel do num_teams(nteams) num_threads(nthreads) map(tofrom: x) ... !$omp end target teams distribute parallel do で囲み, ループをGPU上で並列実行させよ. (結果 x をCPUに戻して検算するので map(tofrom: x) が必要)
  do i = 1, m
     x(i) = lin_rec(0.99d0, real(i, 8), 1.0d0, n)
  end do
  ! TODO: 上で始めた target teams distribute parallel do 領域を閉じる (!$omp end target teams distribute parallel do).
  ! 計測終了
  t1 = omp_get_wtime()
  dt = t1 - t0                  ! sec
  ! 答え確認 (x(i) = 100 * i くらいのはず)
  err = 0
  do i = 1, m
     if (abs(x(i) - 100 * i) > 1.0d-3) then
        print "(a,i3,a,f9.3)", "x[", i, "] = ", x(i)
        err = err + 1
     end if
  end do
  if (err == 0) print "(a)", "OK"
  flops = 2.d0 * real(m, 8) * real(n, 8)
  print "(a,f7.3,a)", "elapsed    : ", dt, "  sec"
  print "(a,f7.3,a)", "elapsed/m  : ", dt / m * 1e3, " msec"
  print "(a,f7.3,a)", "elapsed/n  : ", dt / n * 1e9, " nsec"
  print "(a,f7.3,a)", "elapsed/mn : ", dt / (m * n) * 1e9, " nsec"
  print "(a,es9.2)",  "flops      : ", flops
  print "(f7.3,a)", flops / dt * 1e-9, " GFLOPS"
end program gpu_speedup

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu gpu_speedup.f90 -o gpu_speedup_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gpu_speedup_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:gpu_speedup.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gpu_speedup.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gpu_speedup.cpp}